# Efficient multiscale segmentation (simulation)

Compare a dense segmentation decoder with an original efficient decoder using parallel 3×3 and 5×5 depthwise convolutions and channel gating. Both use the same encoder, skip fusion, BCE plus soft Dice objective and validation-based epoch selection. Train from scratch for nine epochs for each of three seeds. The experiment takes the multiscale depthwise and attention ideas from EMCAD and uses a smaller custom architecture. Generate 480 RGB phantoms with lesion masks, background texture and variable contrast per seed.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'multiscale-efficient-seg-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'torch==2.14.1']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    rng = np.random.default_rng(seed)\n    u, v = np.mgrid[-1:1:40j, -1:1:40j]\n    x = []\n    y = []\n    for _ in range(480):\n        a, b = rng.uniform(-0.5, 0.5, 2)\n        mask = ((u - a) / rng.uniform(0.15, 0.55)) ** 2 + (\n            (v - b) / rng.uniform(0.14, 0.49)\n        ) ** 2 < 1\n        stripes = 0.11 * np.sin(u * rng.uniform(5, 12) + v * rng.uniform(2, 7))\n        texture = rng.normal(0.40, 0.20, (40, 40, 3)) + stripes[:, :, None]\n        x.append(np.clip(texture + mask[:, :, None] * [0.17, 0.08, -0.05], 0, 1))\n        y.append(mask)\n    return np.asarray(x), np.asarray(y)\n\n\nimport torch\nfrom sklearn.model_selection import train_test_split\nfrom torch import nn\nfrom torch.nn import functional as F\n\ntorch.set_num_threads(2)\n\n\nclass Segmenter(nn.Module):\n    def __init__(self, efficient):\n        super().__init__()\n        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU())\n        self.encoder = nn.Sequential(\n            nn.MaxPool2d(2), nn.Conv2d(16, 24, 3, padding=1), nn.ReLU()\n        )\n        self.efficient = efficient\n        self.fusion = nn.Sequential(nn.Conv2d(40, 24, 1), nn.ReLU())\n        if efficient:\n            self.branches = nn.ModuleList(\n                nn.Conv2d(24, 24, k, padding=k // 2, groups=24) for k in (3, 5)\n            )\n            self.mix = nn.Conv2d(48, 24, 1)\n            self.gate = nn.Sequential(\n                nn.AdaptiveAvgPool2d(1),\n                nn.Conv2d(24, 8, 1),\n                nn.ReLU(),\n                nn.Conv2d(8, 24, 1),\n                nn.Sigmoid(),\n            )\n        else:\n            self.context = nn.Sequential(\n                nn.Conv2d(24, 24, 3, padding=1),\n                nn.ReLU(),\n                nn.Conv2d(24, 24, 3, padding=1),\n            )\n        self.head = nn.Conv2d(24, 1, 1)\n\n    def forward(self, x):\n        skip = self.stem(x)\n        z = self.encoder(skip)\n        if self.efficient:\n            z = z + self.mix(\n                torch.cat([branch(z) for branch in self.branches], dim=1)\n            ) * self.gate(z)\n        else:\n            z = z + self.context(z)\n        z = F.interpolate(z, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n        return self.head(self.fusion(torch.cat([z, skip], dim=1)))\n\n\ndef dice(pred, y):\n    p = pred >= 0\n    intersection = (p * y).sum((1, 2, 3))\n    return (\n        ((2 * intersection + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1))\n        .mean()\n        .item()\n    )\n\n\ndef run(seed):\n    x, y = dataset(seed)\n    x = torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)\n    y = torch.tensor(y, dtype=torch.float32)[:, None]\n    train, rest = train_test_split(np.arange(len(y)), test_size=0.36, random_state=seed)\n    valid, test = np.array_split(rest, 2)\n    results = {}\n    for tag in ["dense", "depthwise"]:\n        torch.manual_seed(seed)\n        model = Segmenter(tag == "depthwise")\n        optim = torch.optim.AdamW(model.parameters(), lr=0.003)\n        best = -1\n        for _ in range(9):\n            model.train()\n            for idx in torch.randperm(len(train)).split(24):\n                ids = train[idx.numpy()]\n                p = model(x[ids])\n                target = y[ids]\n                f = p.sigmoid()\n                overlap = (2 * (f * target).sum() + 1) / (f.sum() + target.sum() + 1)\n                loss = F.binary_cross_entropy_with_logits(p, target) + 1 - overlap\n                optim.zero_grad()\n                loss.backward()\n                optim.step()\n            model.eval()\n            with torch.no_grad():\n                val = dice(model(x[valid]), y[valid])\n            if val > best:\n                best = val\n                state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(state)\n        folder = ROOT / "checkpoints"\n        folder.mkdir(exist_ok=True)\n        torch.save(state, folder / f"{tag}-{seed}.pt")\n        with torch.no_grad():\n            value = dice(model(x[test]), y[test])\n        results[tag] = {\n            "dice": value,\n            "parameters": sum(p.numel() for p in model.parameters()),\n            "test_images": len(test),\n        }\n    return results\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())